# Lopez-Salido, Stein & Zakrajsek (2017): Replication Walkthrough
### *Credit-Market Sentiment and the Business Cycle*, QJE 132(3)

This notebook walks through our replication of the paper's core results on
annual U.S. data, 1929-2015. The thesis: **elevated credit-market
sentiment** — unusually narrow credit spreads and a high high-yield issuance
share — **predicts a subsequent slowdown in real activity**, because frothy
credit conditions mean-revert.

| Exhibit | What it shows | Status |
|---|---|---|
| Figure I | The Baa-Treasury credit spread, 1929-2015 | done |
| Table I  | Credit spread changes vs. equity returns as growth predictors | done |
| Table II | Two-step sentiment forecast of growth | done |
| Figure II | Fitted sentiment vs. realized growth | done |

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd().parent / "src"))
from settings import config

PROCESSED_DATA_DIR = Path(config("PROCESSED_DATA_DIR"))
OUTPUT_DIR = Path(config("OUTPUT_DIR"))

## Data

All series come from the pipeline (`doit pull_fred process_fred_data`,
`doit pull_shiller`). We read the processed files directly rather than
re-pulling.

In [ ]:
annual = pd.read_parquet(PROCESSED_DATA_DIR / "fred_final_series_annual.parquet")
monthly = pd.read_parquet(PROCESSED_DATA_DIR / "fred_final_series_monthly.parquet")
print("Annual columns:", annual.columns.tolist())
annual.head()

## Figure I — The Baa-Treasury credit spread

The paper's opening exhibit: the credit spread is strongly countercyclical,
spiking into every recession. Its *change* is the key predictor in Table I.

In [ ]:
spread = monthly["BAA_Treasury_spread"].dropna()
spread = spread[(spread.index >= "1929") & (spread.index <= "2015")]

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(spread.index, spread.values, color="#1f4e79", lw=1.1)
ax.set_title("Figure I: Baa-Treasury Credit Spread, 1929-2015")
ax.set_ylabel("Percentage points")
plt.show()

## Table I — Credit beats equities

Table I forecasts next-year real GDP-per-capita growth ($\Delta y_{t+1}$)
with the current-year change in the credit spread ($\Delta s_t$) and the S&P
500 total return ($r^{SP}_t$), Newey-West errors. Three columns: credit only,
equity only, and both plus rate/inflation controls.

The **no-dummies 1929-2015** table is the faithful match to the published QJE
Table I. Our replication reproduces it almost coefficient-for-coefficient:

| | Published QJE | Our replication |
|---|---|---|
| col 1 $\Delta s_t$ | -1.997 | -1.958 |
| col 2 $r^{SP}_t$ | 0.081 | 0.075 |
| col 3 $\Delta s_t$ | -2.061 | -2.124 |
| col 3 $r^{SP}_t$ | 0.029 | 0.022 |

Column 3 is the paper's headline: with both predictors in, **credit stays
strongly significant while the equity coefficient collapses** — the credit
market forecasts the economy better than the stock market.

In [ ]:
import replicate_table_1 as t1

df = t1.build_panel()
res = t1.run_regression(
    df,
    ["d_credit_spread", "sp_return", "d_treasury_3mo",
     "d_treasury_10yr", "CPI_inflation", "gdp_pc_growth"],
    t1.REP_START, t1.REP_END,
)
print(res.summary())

The generated LaTeX tables are written to `_output/` by
`doit replicate_table_1`. `table_1_replication_nodummies.tex` matches the
published paper; the `dummies` variants match the working-paper spec, and the
`extended` variants carry the sample through 2023 (the Shiller endpoint).

In [ ]:
print((OUTPUT_DIR / "table_1_replication.tex").read_text())

## Table II & Figure II — the sentiment two-step

Table II is the analytical core. A *first stage* forecasts the change in the
credit spread from twice-lagged sentiment (the log high-yield issuance share
$\ln[\mathrm{HYS}]_{t-2}$ and the spread level $s_{t-2}$), and a separate
auxiliary regression forecasts the equity return from $\ln[P/E10]_{t-2}$. The
*second stage* then regresses growth on those **fitted** values — so the
right-hand side contains only the predictable, sentiment-driven component of
each market's move.

The LSZ signs are $a_1 > 0$ (froth forecasts a widening spread) and $a_2 < 0$
(wide spreads mean-revert); the headline second-stage result is that fitted
spread widening forecasts *lower* growth.

In [ ]:
import replicate_table_2 as t2

df2 = t2.build_panel()
res2 = t2.run_table_2(df2, t2.REP_START, t2.REP_END)

aux = res2["aux_spread"]
print("First stage  Δs_t = a0 + a1·lnHYS_{t-2} + a2·s_{t-2}")
print(f"  a1 (froth)  {aux.params['ln_hys_lag2']:+.3f}  (p={aux.pvalues['ln_hys_lag2']:.3f})")
print(f"  a2 (level)  {aux.params['spread_lag2']:+.3f}  (p={aux.pvalues['spread_lag2']:.3f})")
print(f"  R2 = {aux.rsquared:.3f}\n")

for col in ("col1", "col2", "col3", "col4"):
    res = res2[col]
    terms = ", ".join(f"{v}={res.params[v]:+.3f}" for v in t2.COLUMN_REGRESSORS[col])
    print(f"  {col}: N={int(res.nobs)} R2={res.rsquared:.3f}  {terms}")

In [ ]:
import matplotlib.pyplot as plt

import replicate_figure_2 as f2

fig = f2.plot_figure_2(df2, t2.REP_START, t2.REP_END)
plt.show()

## Summary

Our pipeline reproduces the published QJE Figure I and Table I closely —
including the credit-beats-equities result in Table I column (3) — and the
Table II sentiment two-step recovers both first-stage signs along with the
negative second-stage coefficient on fitted spread widening. Figure II shows
that relationship against realized growth with influential years flagged.

The Aaa-spread extension of all four exhibits is walked through in
`03_extension.ipynb`, and an out-of-sample application of the sentiment signal
to the 2020-2022 cycle is in `04_case_study.ipynb`.